# BCS 3101: Basics of Machine Learning
## Assignment 2 – Notebook 5

**Student Name:** ARINDA ELIZABETH  
**Registration Number:** 2024/A/KCS/3099/G/F  
**Group Project:** Predicting Monthly Maize and Beans Prices in Selected Ugandan Markets  

This notebook covers **Stage 9: Data Pre-processing IV – Reduction, Splitting & Model Readiness** from the companion guide.

Data Reduction & Splitting is worth **10%** on the marking rubric.  
We follow the exact steps and the reasoning the lecturer expects.

---
## Load and prepare the data

We start from the filtered dataset prepared in Notebook 1.  
We re-apply the key encoding steps so this notebook can run on its own even if the intermediate CSV files from Notebooks 3 and 4 are not yet present.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_style('whitegrid')

# Load the filtered dataset from Notebook 1
df = pd.read_csv('uganda_maize_beans_selected_markets.csv')

print("Data loaded.")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

In [ ]:
# One-hot encode the market name (same decision as Notebook 3)
df = pd.get_dummies(df, columns=['mkt_name'], drop_first=True, dtype=int)

# Select the columns we will use
feature_candidates = [
    'year', 'month', 'lat', 'lon',
    'c_oil', 'c_salt', 'c_food_price_index',
    'inflation_maize', 'inflation_beans',
    'trust_maize', 'trust_beans',
    'data_coverage', 'data_coverage_recent', 'index_confidence_score'
]

# Add any one-hot market columns that exist
market_cols = [c for c in df.columns if c.startswith('mkt_name_')]
feature_candidates = feature_candidates + market_cols

# Keep only columns that are actually present
feature_cols = [c for c in feature_candidates if c in df.columns]

# Targets
target_cols = ['c_maize', 'c_beans']

# Drop rows that still have missing values in the columns we need
df_clean = df[feature_cols + target_cols].dropna().copy()

print("Shape after dropping rows with missing values in selected columns:")
print(df_clean.shape)
print("\nFeature columns:")
print(feature_cols)
print("\nTarget columns:")
print(target_cols)

---
## 9.1 Feature Selection

The guide says:

> “Not every column adds useful information. Some are redundant (highly correlated with another feature), irrelevant to the target, or simply noise. Reducing feature count fights the curse of dimensionality…”

We use a simple **filter method**: look at the correlation of each feature with the two targets and also check for highly correlated pairs among the features themselves.

In [ ]:
# Correlation of each feature with the two targets
corr_with_targets = df_clean[feature_cols + target_cols].corr()[target_cols].drop(target_cols)
print("Correlation of features with the targets:")
print(corr_with_targets.round(3))

In [ ]:
# Heatmap of correlations among the numeric features (excluding one-hot for clarity)
numeric_features = [c for c in feature_cols if not c.startswith('mkt_name_')]

plt.figure(figsize=(10, 8))
corr_matrix = df_clean[numeric_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation heatmap of numeric features')
plt.tight_layout()
plt.show()

### Decision on feature selection

From the correlations we keep the features that show a reasonable link with the targets or that carry useful market/time information.  
We drop any feature that is almost constant or that is extremely highly correlated with another feature we already keep (to avoid redundancy).

For this project we keep all the selected feature columns.  
The number of features is still modest (well under 20), so the risk of the curse of dimensionality is low.

---
## 9.2 Dimensionality Reduction with PCA

The guide explains PCA as a way to re-express numeric features as a smaller set of uncorrelated components that capture as much variance as possible.

> “Because PCA is a distance- and variance-based technique, features MUST be standardised before applying it…”

We first standardise the features, then fit PCA and look at the cumulative variance explained.

In [ ]:
# Standardise the features (required before PCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[feature_cols])

# Fit PCA that keeps enough components to explain 95% of the variance
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print(f"Original number of features: {len(feature_cols)}")
print(f"Number of PCA components kept for 95% variance: {pca.n_components_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print("\nVariance explained by each component:")
print(np.round(pca.explained_variance_ratio_, 3))

In [ ]:
# Scree plot
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(pca.explained_variance_ratio_)+1),
        pca.explained_variance_ratio_, alpha=0.7, label='Individual')
plt.plot(range(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker='o', color='red',
         label='Cumulative')
plt.axhline(y=0.95, color='green', linestyle='--', label='95% threshold')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.title('PCA Scree Plot')
plt.legend()
plt.tight_layout()
plt.show()

### Decision on PCA

The guide says:

> “PCA is not always relevant, especially with a modest number of features that are already largely uncorrelated and individually interpretable. If you judge PCA unnecessary, say so explicitly with a reason… The rubric explicitly credits a reasoned decision to omit a step over a silent absence.”

**Our decision:** We will **not** use the PCA-transformed features for the final model.

**Reason:**  
We have fewer than 20 features. Most of them (year, month, market indicators, a few price and inflation series) are easy to interpret.  
Keeping the original features makes the later analysis and the report clearer.  
PCA would mix the variables into components that are harder to explain to a non-technical reader.  
We therefore keep the original (standardised) features and treat the PCA results only as an exploration of how much variance the data contains.

---
## 9.3 Handling Class Imbalance

This section of the guide is written mainly for classification problems (where one class can dominate another).

Our problem is **regression** (we predict continuous prices).  
There is no class label, so class-imbalance techniques such as SMOTE or random oversampling do not apply.

We simply state this fact so the examiner sees we read the stage and made a reasoned judgement.

In [ ]:
print("Problem type: Regression (continuous targets)")
print("Class-imbalance handling: NOT APPLICABLE")
print("Reason: There are no class labels to balance.")

---
## 9.4 The Train/Test Split

The guide stresses:

> “A model must be evaluated on data it has never seen during training… The standard practice is to split your data before any model-fitting step… into a training partition and a held-out test partition.”

We use an 80/20 split with a fixed random_state so the result is reproducible.

In [ ]:
# Prepare X and y
X = df_clean[feature_cols]
y_maize = df_clean['c_maize']
y_beans = df_clean['c_beans']

# Split for maize target
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_maize, test_size=0.2, random_state=42
)

# Split for beans target (same random_state so the rows match)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_beans, test_size=0.2, random_state=42
)

print("Maize split:")
print(f"  Training set: {X_train_m.shape[0]} rows")
print(f"  Test set:     {X_test_m.shape[0]} rows")
print("\nBeans split:")
print(f"  Training set: {X_train_b.shape[0]} rows")
print(f"  Test set:     {X_test_b.shape[0]} rows")

### Correct order of scaling (important)

The guide is very clear:

> “Always split first, then fit pre-processing steps on the training partition only.”

We now fit the StandardScaler **only on the training features** and then transform both train and test sets.

In [ ]:
# Fit scaler on training data only
scaler_final = StandardScaler()
X_train_m_scaled = scaler_final.fit_transform(X_train_m)
X_test_m_scaled  = scaler_final.transform(X_test_m)   # transform only – do not re-fit

# Same scaler can be used for the beans split because the feature matrix is identical
X_train_b_scaled = scaler_final.transform(X_train_b)
X_test_b_scaled  = scaler_final.transform(X_test_b)

print("Scaling done correctly:")
print("  Scaler was fitted on the training set only.")
print("  Test set was transformed with the same scaler (no data leakage).")
print(f"\nTraining features shape (maize): {X_train_m_scaled.shape}")
print(f"Test features shape (maize):     {X_test_m_scaled.shape}")

In [ ]:
# Save the split data so the final notebook can use it
np.savez('train_test_split_maize.npz',
         X_train=X_train_m_scaled, X_test=X_test_m_scaled,
         y_train=y_train_m.values, y_test=y_test_m.values)

np.savez('train_test_split_beans.npz',
         X_train=X_train_b_scaled, X_test=X_test_b_scaled,
         y_train=y_train_b.values, y_test=y_test_b.values)

print("Train/test splits saved as .npz files.")
print("These files are ready for any later modelling step.")

---
## Summary of Stage 9 decisions (ready for the report)

| Step | Decision | Reason |
|------|----------|--------|
| Feature selection | Kept the selected numeric + one-hot market features | Modest number of features; each has a clear meaning |
| PCA | Explored but not used for modelling | Features are already interpretable; PCA would reduce clarity |
| Class imbalance | Not applicable | Regression problem, no class labels |
| Train/test split | 80/20 with random_state=42 | Standard, reproducible split |
| Scaling order | Fit on training set only, then transform test set | Prevents data leakage (guide requirement) |

All of the above follow Stage 9 of the companion guide and will be written into the final report with the exact numbers shown in this notebook.

---
## End of Notebook 5

### What we finished
- Prepared a clean feature matrix and two target series
- Examined correlations and decided which features to keep
- Ran PCA, inspected the scree plot, and decided **not** to use the PCA components for modelling
- Stated that class-imbalance methods are not applicable
- Performed a proper 80/20 train-test split
- Fitted the scaler only on the training data (no leakage)
- Saved the split arrays for later use

### What comes next
Notebook 6 (GUMISIRIZA AMBROSE) will perform the full **Exploratory Data Analysis (Stage 10)** – the largest single criterion (20%).

**Reminder from the marking criteria**  
Data Reduction & Splitting is worth 10%. The examiner looks for a clear, reasoned decision on PCA and for the correct train-then-transform order of scaling.